In [ ]:
from scapyter.domain.analysis.correlation.cpa import CpaCorrelation
from scapyter.domain.analysis.correlation.service import CorrelationService
from scapyter.domain.leakage.leakage import SboxOutputLeakageModel
from scapyter.domain.value_object import RangeParameters, Range, DataSource, KeyByteGuesses
from scapyter.ui.correlation_plotter import CorrelationPlotter
from scapyter.ui.key_rank_visualizer import KeyRankVisualizer
from scapyter.infrastructure.h5_project_file_reader import H5ProjectFileReader

In [ ]:
from scapyter.ui.trace_plotter import TracePlotter

file_path = "data/smart-card-project/smart_card_project.sx"

trace_repo = H5ProjectFileReader(file_path)
trace_plotter = TracePlotter(trace_repo)
trace_plotter.plot_single(index=1)

In [ ]:
range_parameter = RangeParameters(
    trace_range=Range(0, 125),
    trace_sample_range=Range(14000, 44000),
)

leakage_model = SboxOutputLeakageModel()

results = []
for i in range(16):
    snr = CpaCorrelation()
    key_byte_guesses = KeyByteGuesses.with_correct_and_random_key_bytes(correct_key_byte=i, num_random_key_bytes=5)
    result = CorrelationService(
        byte_location=i,
        range_parameters=range_parameter,
        leakage_model=leakage_model,
        correlation=snr,
        project_file_reader=trace_repo,
        data_source=DataSource.PLAINTEXT,
        key_byte_guesses=key_byte_guesses,

    ).run()
    results.append(result)

In [ ]:
visualizer = KeyRankVisualizer(results)

# 2. Display the table
# This will show the top 5 candidates for every byte
visualizer.display_rank_table(top_n=5)

# 3. Print the final guessed key hex string
guessed_key = visualizer.get_full_key_guess()
print(f"Guessed Key: {guessed_key.hex().upper()}")

In [ ]:
plotter = CorrelationPlotter(results)
plotter.plot(0)

In [ ]:
plotter.plot(1)

In [ ]:

from scapyter.domain.analysis.snr.snr_service import SnrService
from scapyter.domain.analysis.snr.snr import ProgressiveSnr

trace_sample_range = Range(14000, 44000)
range_parameter = RangeParameters(
    trace_range=Range(0, 125),
    trace_sample_range=trace_sample_range,
)

leakage_model = SboxOutputLeakageModel()

results = []
for i in range(16):
    snr = ProgressiveSnr()
    result = SnrService(
        byte_location=i,
        range_parameters=range_parameter,
        leakage_model=leakage_model,
        snr=snr,
        project_file_reader=trace_repo,
        data_source=DataSource.PLAINTEXT,
        known_key_byte=i
    ).run()
    results.append((i, result))

In [ ]:
from scapyter.ui.snr_plotter import SnrPlotter

plotter = SnrPlotter(results[1])
plotter.plot()